In [1]:
print('yellou')

yellou


In [2]:
from pathlib import Path
import pandas as pd

# 1. Ruta relativa al archivo procesado generado por run_pipeline.py
data_path = Path("data/latest_event.json")

# 2. Cargar directamente el DataFrame ya unido
df_merged = pd.read_json(data_path)

# Verificación rápida
print(f"Cargados {len(df_merged)} matchups exitosamente.")
df_merged[["matchup", "fighter_1", "fighter_2"]].head()

Cargados 12 matchups exitosamente.


,matchup,fighter_1,fighter_2
0,Brendan Allen vs Christian Leroy Duncan,Brendan Allen,Christian Leroy Duncan
1,Jai Herbert vs Matheus Camilo,Jai Herbert,Matheus Camilo
2,Lupita Godinez vs Ketlen Souza,Lupita Godinez,Ketlen Souza
3,Kai Kamaka III vs Andre Fili,Kai Kamaka III,Andre Fili
4,Gerald Meerschaert vs Julius Walker,Gerald Meerschaert,Julius Walker


In [3]:
from IPython.display import display
import ipywidgets as widgets


def formatear_historial(historial):
  """Extrae un resumen legible usando saltos de línea HTML para que luzca ordenado en la celda."""
  if not isinstance(historial, list) or len(historial) == 0:
    return "No data"

  resumenes = []
  for pelea in historial:
    resumen = (
        f"<b>{pelea.get('result', '')}</b> {pelea.get('opponent', '')}"
        f" ({pelea.get('method', '')}) - R{pelea.get('round', '')}"
        f" ({pelea.get('time', '')})<br>{pelea.get('event', '')}"
        f" ({pelea.get('date', '')})"
    )
    resumenes.append(resumen)
  return "<br>".join(resumenes)


def mostrar_comparativa(matchup_seleccionado):
  row = df_merged[df_merged["matchup"] == matchup_seleccionado].iloc[0]

  f1_name = row["fighter_1"]
  f2_name = row["fighter_2"]

  stats = [
      col.replace("_f1", "")
      for col in df_merged.columns
      if col.endswith("_f1") and col not in ["fighter_f1", "matchup_f1"]
  ]

  datos_comparativos = []
  for stat in stats:
    val_f1 = row.get(f"{stat}_f1", "N/A")
    val_f2 = row.get(f"{stat}_f2", "N/A")

    if stat == "sherdog_recent_fights":
      val_f1 = formatear_historial(val_f1)
      val_f2 = formatear_historial(val_f2)
    elif stat == "last_5" and isinstance(val_f1, list):
      val_f1 = "-".join(val_f1)
      val_f2 = "-".join(val_f2) if isinstance(val_f2, list) else val_f2

    datos_comparativos.append({
        "Estadística": (
            stat.replace("sherdog_", "").replace("_", " ").title()
        ),
        f1_name: val_f1,
        f2_name: val_f2,
    })

  df_comp = pd.DataFrame(datos_comparativos).set_index("Estadística")

  # Muestra la tabla formateada en HTML con soporte de saltos de línea
  display(
      df_comp.style.set_properties(**{
          "text-align": "left",
          "white-space": "normal",
          "max-width": "400px",
      })
  )


# Crear el menú desplegable con la lista de enfrentamientos
lista_matchups = df_merged["matchup"].dropna().unique().tolist()
dropdown = widgets.Dropdown(
    options=lista_matchups,
    description="Matchup:",
    disabled=False,
    layout={"width": "max-content"},
)

# Conectar el menú a la función interactiva
widgets.interact(mostrar_comparativa, matchup_seleccionado=dropdown)

interactive(children=(Dropdown(description='Matchup:', layout=Layout(width='max-content'), options=('Brendan A…

<function __main__.mostrar_comparativa(matchup_seleccionado)>